# Experiment 6: Transfer Across Backbone Depth

Tests whether the meta-encoder framework generalizes across backbone architectures
of varying depth: ResNet18 (8 layers), ResNet34 (16 layers), ResNet50 (16 layers).

Evaluates Criteria 1 (R²) and 2 (Spearman ρ) for each backbone.

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import copy
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder
from losses.info_loss import InfoLoss
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.metrics import rich_profile_reconstruction_r2, geometric_consistency

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# One config + checkpoint per backbone
# phase1.yaml uses resnet18; override arch for others
# TODO: Train backbone-specific models and update checkpoint paths
backbone_experiments = {
    'resnet18': {
        'checkpoint': CKPT_DIR + '/phase1/best.pt',
    },
    'resnet34': {
        'checkpoint': CKPT_DIR + '/backbone_transfer/resnet34/best.pt',
    },
    'resnet50': {
        'checkpoint': CKPT_DIR + '/backbone_transfer/resnet50/best.pt',
    },
}

In [ ]:
# Load base config once; override arch per experiment
with open(CONFIG_DIR + '/phase1.yaml') as f:
    base_config = yaml.safe_load(f)

results   = {}
MAX_PAIRS = 50_000

for arch, exp in backbone_experiments.items():
    print(f'\n=== {arch.upper()} ===')
    if not os.path.exists(exp['checkpoint']):
        print(f'  Checkpoint not found: {exp["checkpoint"]}')
        continue

    config = copy.deepcopy(base_config)
    config['model']['arch']    = arch
    config['data']['data_dir'] = DATA_DIR

    flow_cfg = config['model'].get('flow_compression', {})

    backbone = FrozenBackbone(
        arch=arch,
        num_classes=config['model']['num_classes'],
        pretrained=config['model']['pretrained'],
        grid_size=flow_cfg.get('grid_size', 4),
        flow_dim=flow_cfg.get('flow_dim', 256),
    ).to(device)

    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        projection_dim=config['model']['meta_encoder']['projection_dim'],
        n_heads=config['model']['meta_encoder']['n_heads'],
        n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
        dropout=config['model']['meta_encoder']['dropout']
    ).to(device)

    info_loss = InfoLoss(
        layer_dims=backbone.layer_dims,
        projection_dim=config['model']['meta_encoder']['projection_dim'],
        hidden_dim=config['model']['regressor']['hidden_dim'],
    ).to(device)

    ckpt = torch.load(exp['checkpoint'], map_location=device, weights_only=False)
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    info_loss.load_state_dict(ckpt['info_loss_state'])
    meta_encoder.eval()
    info_loss.eval()

    _, val_loader = get_standard_loaders(
        data_dir=DATA_DIR,
        batch_size=config['data'].get('batch_size', 256),
        num_workers=0,
        augment=False,
        download=False,
    )

    analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
    data     = analyzer.collect_representations(max_samples=2000)

    N = data['z_list'][0].shape[0]
    idx_a, idx_b = torch.triu_indices(N, N, offset=1)
    if idx_a.shape[0] > MAX_PAIRS:
        perm  = torch.randperm(idx_a.shape[0])[:MAX_PAIRS]
        idx_a, idx_b = idx_a[perm], idx_b[perm]

    L = len(data['z_list'])

    # Scalar flow similarities for Criterion 2
    flow_sims_np = np.zeros((idx_a.shape[0], L))
    for l in range(L):
        flow_sims_np[:, l] = (
            data['flow_targets'][l][idx_a].numpy() * data['flow_targets'][l][idx_b].numpy()
        ).sum(axis=1)

    z_sims_np = np.zeros((idx_a.shape[0], L))
    for l in range(L):
        z_sims_np[:, l] = (
            data['z_list'][l][idx_a].numpy() * data['z_list'][l][idx_b].numpy()
        ).sum(axis=1)

    # Criterion 1: flow co-activation R²
    flow_coact = CircuitAnalyzer.compute_pair_rich_profiles(data['flow_targets'], idx_a, idx_b)
    preds = []
    with torch.no_grad():
        for l in range(L):
            z_prod = data['z_list'][l][idx_a] * data['z_list'][l][idx_b]
            preds.append(info_loss.regressors[l](z_prod.to(device)).cpu().numpy())
    r2_result = rich_profile_reconstruction_r2(preds, [t.numpy() for t in flow_coact])

    # Criterion 2: geometric consistency (Spearman rho vs flow similarity)
    gc_result = geometric_consistency(z_sims_np, flow_sims_np, L)

    results[arch] = {
        'r2': r2_result['r2'], 'mean_rho': gc_result['mean_rho'],
        'r2_passes': r2_result['passes'], 'gc_passes': gc_result['passes'],
    }
    print(f"  R² = {r2_result['r2']:.4f}  (passes: {r2_result['passes']})")
    print(f"  Mean ρ = {gc_result['mean_rho']:.4f}  (passes: {gc_result['passes']})")

In [ ]:
# Grouped bar chart
if results:
    archs = list(results.keys())
    r2s   = [results[a]['r2']       for a in archs]
    rhos  = [results[a]['mean_rho'] for a in archs]

    x     = np.arange(len(archs))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, r2s,  width, label='R\u00b2',          color='steelblue')
    ax.bar(x + width/2, rhos, width, label='Mean Spearman \u03c1', color='coral')
    ax.axhline(0.7,  color='steelblue', linestyle='--', alpha=0.6, label='R\u00b2 target')
    ax.axhline(0.65, color='coral',     linestyle='--', alpha=0.6, label='\u03c1 target')
    ax.set_xticks(x)
    ax.set_xticklabels(archs)
    ax.set_title('Transfer Across Backbone Depth')
    ax.legend()
    plt.tight_layout()
    plt.show()